### decision tree
Implement the ID3, C4.5, and CART decision tree algorithms from scratch (without using any machine learning libraries) for a suitable classification dataset. Develop the complete training procedure by implementing the required attribute selection measures, construct the decision trees, and use the generated models to classify test instances. Compare the three algorithms in terms of classification accuracy, precision, recall, F1-score, tree depth, number of leaf nodes, and training time, and summarize your observations on their performance and the resulting tree structures.


In [1]:
import numpy as np
import pandas as pd


In [2]:
def load_data():
    col_names = ['sepal_length','sepal_width','petal_length','petal_width','type']
    data = pd.read_csv("iris.csv",skiprows = 1,header = None, names = col_names)
    X = data.iloc[:,:-1].values
    Y = data.iloc[:, -1].values.reshape(-1,1)
    return X, Y

def train_test_spilit_scratch(X,Y,test_size = 0.2,random_state=42):
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X)*test_size)
    X_train = X[indices[test_samples:]]
    Y_train = Y[indices[test_samples:]]
    X_test = X[indices[:test_samples]]
    Y_test = Y[indices[:test_samples]]
   
    return X_train,Y_train,X_test,Y_test



    
    

In [3]:
class Node:
    def __init__(self,feature_index = None,information_gain=None,left = None,right = None,threshold = None,value=None):

        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.information_gain = information_gain

        # leaf Node
        self.value = value

class DecisionTreeClassifier():
    def __init__(self, min_samples_spilit = 2, max_depth = 2, criterion = "gini"):
        self.root = None
        self.min_samples_spilit = min_samples_spilit
        self.max_depth = max_depth
        self.criterion = criterion
    
    def build_tree(self,dataset,curr_depth = 0):
        X,Y = dataset[:,-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)

        if num_samples >= self.min_samples_spilit and curr_depth <= self.max_depth:
            best_split = self.get_best_split(dataset,num)
    
    def get_best_split(self,dataset, num_samples, num_features):
        best_split = {}
        max_info_gain = -float("inf")

        

In [4]:
import numpy as np
import pandas as pd
import time

def calc_entropy(y):
    probs = np.unique(y, return_counts=True)[1] / len(y)
    return -np.sum(probs * np.log2(probs))

def calc_gini(y):
    probs = np.unique(y, return_counts=True)[1] / len(y)
    return 1 - np.sum(probs**2)

def evaluate_attribute(X, Y, attr_idx, method):
    parent_score = calc_entropy(Y) if method in ['ID3', 'C4.5'] else calc_gini(Y)
    
    outcomes, counts = np.unique(X[:, attr_idx], return_counts=True)
    weighted_score, split_info = 0, 0
    
    for j, count in zip(outcomes, counts):
        subset_Y = Y[X[:, attr_idx] == j]
        weight = count / len(Y)
        
        if method in ['ID3', 'C4.5']:
            weighted_score += weight * calc_entropy(subset_Y)
            split_info -= weight * np.log2(weight + 1e-9)
        else:
            weighted_score += weight * calc_gini(subset_Y)
            
    gain = parent_score - weighted_score
    if method == 'C4.5': return gain / split_info if split_info > 0 else 0
    return gain

def generate_decision_tree(D_X, D_Y, attribute_list, method='ID3'):
    classes, counts = np.unique(D_Y, return_counts=True)
    if len(classes) == 1:
        return classes[0]
    
    majority_class = classes[np.argmax(counts)]
    
    if len(attribute_list) == 0:
        return majority_class
    
    scores = [evaluate_attribute(D_X, D_Y, attr, method) for attr in attribute_list]
    best_attr = attribute_list[np.argmax(scores)]
    
    node = {best_attr: {}}
    
    new_attr_list = [a for a in attribute_list if a != best_attr]
    
    outcomes = np.unique(D_X[:, best_attr])
    for j in outcomes:
        subset_indices = D_X[:, best_attr] == j
        Dj_X, Dj_Y = D_X[subset_indices], D_Y[subset_indices]
        
        if len(Dj_Y) == 0:
            node[best_attr][j] = majority_class
        else:
            node[best_attr][j] = generate_decision_tree(Dj_X, Dj_Y, new_attr_list, method)
            
    return node

def predict(tree, x, default_class):
    if not isinstance(tree, dict): return tree
    
    attr = list(tree.keys())[0]
    val = x[attr]
    
    if val in tree[attr]:
        return predict(tree[attr][val], x, default_class)
    return default_class

def get_stats(tree, depth=0):
    if not isinstance(tree, dict): return depth, 1
    
    depths, leaves = zip(*[get_stats(child, depth + 1) for child in list(tree.values())[0].values()])
    return max(depths), sum(leaves)

def simple_metrics(y_true, y_pred):
    y_pred = np.array(y_pred) # <--- THE FIX
    acc = np.mean(y_true == y_pred)
    classes = np.unique(y_true)
    prec, rec, f1 = [], [], []
    
    for c in classes:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        
        prec.append(p)
        rec.append(r)
        f1.append(2 * p * r / (p + r) if p + r > 0 else 0)
        
    return acc, np.mean(prec), np.mean(rec), np.mean(f1)

data = pd.read_csv("iris.csv", skiprows=1, header=None)
X = data.iloc[:, :-1].values
Y = data.iloc[:, -1].values

X_binned = np.digitize(X, bins=[np.percentile(X, 33), np.percentile(X, 66)])

np.random.seed(41)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]
X_train, Y_train = X_binned[train_idx], Y[train_idx]
X_test, Y_test = X_binned[test_idx], Y[test_idx]

majority_lbl = np.unique(Y_train)[np.argmax(np.unique(Y_train, return_counts=True)[1])]

for method in ['ID3', 'C4.5', 'CART']:
    print(f"\n--- {method} ---")
    start = time.time()
    
    attribute_list = list(range(X_train.shape[1]))
    tree = generate_decision_tree(X_train, Y_train, attribute_list, method=method)
    
    train_time = time.time() - start
    
    Y_pred = [predict(tree, x, majority_lbl) for x in X_test]
    acc, prec, rec, f1 = simple_metrics(Y_test, Y_pred)
    depth, leaves = get_stats(tree)
    
    print(f"Time: {train_time:.5f}s | Depth: {depth} | Leaves: {leaves}")
    print(f"Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | F1: {f1:.2f}")


--- ID3 ---
Time: 0.01383s | Depth: 5 | Leaves: 5
Acc: 0.90 | Prec: 0.91 | Rec: 0.90 | F1: 0.90

--- C4.5 ---
Time: 0.00951s | Depth: 5 | Leaves: 5
Acc: 0.90 | Prec: 0.91 | Rec: 0.90 | F1: 0.90

--- CART ---
Time: 0.00780s | Depth: 5 | Leaves: 5
Acc: 0.90 | Prec: 0.91 | Rec: 0.90 | F1: 0.90
